# 04 Measure Clustering

Assignment step 7: group final measures into statistical domains while preserving unclustered/noise measures.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Clustering Summary


In [2]:
metrics = read_json("report/clustering_metrics.json")
{
    "measure_count": metrics.get("measure_count"),
    "cluster_count": metrics.get("cluster_count"),
    "coverage": metrics.get("coverage"),
    "unclustered_count": metrics.get("unclustered_count"),
    "baseline": metrics.get("baseline"),
    "validation": metrics.get("validation"),
    "manual_review": metrics.get("manual_review"),
}

{'measure_count': 2894,
 'cluster_count': 216,
 'coverage': 0.8700760193503801,
 'unclustered_count': 376,
 'baseline': {'cluster_count': 48,
  'distance_threshold': 'configured',
  'method': 'agglomerative'},
 'validation': {'failures': [], 'passed': True},
 'manual_review': {'completed_count': 0,
  'review_sample': 'outputs\\clustering\\run_e1c656105fd6bcd3c088\\manual_cluster_review_sample.csv',
  'sample_count': 217,
  'status': 'pending'}}

## Domain Distribution


In [3]:
read_json("report/clustering_metrics.json").get("domain_distribution", {})

{'agriculture, forestry, and fisheries': 145,
 'cross-domain or other': 1333,
 'economy and finance': 493,
 'education': 60,
 'environment and energy': 48,
 'government, justice, and public administration': 6,
 'health': 70,
 'industry, trade, and services': 45,
 'labour market': 248,
 'population and demography': 361,
 'social conditions and equality': 29,
 'transport': 56}

## Cluster Examples


In [4]:
csv_rows("outputs/measure_clusters.csv", 8)

[{'term_id': 'term_f92aa0607c2b5c538059',
  'term': '1 activity and employment',
  'cluster_id': 'cluster_15e007abd64e',
  'domain': 'labour market',
  'membership_probability': '1.000000',
  'is_representative': 'true',
  'labeling_method': 'embedding_threshold',
  'evidence': '{"classification_run_id": "run_42d33d7967059d3e8f22", "domain_scores": {"agriculture, forestry, and fisheries": 0.6790877284174742, "economy and finance": 0.7015241122992016, "environment and energy": 0.6757921987770852, "industry, trade, and services": 0.7310639385211554, "labour market": 0.7643449314420573}, "source": "hdbscan_pearl_small"}'},
 {'term_id': 'term_ae1f4ef44ee128d00b40',
  'term': '1 activity and employment status',
  'cluster_id': 'cluster_15e007abd64e',
  'domain': 'labour market',
  'membership_probability': '1.000000',
  'is_representative': 'true',
  'labeling_method': 'embedding_threshold',
  'evidence': '{"classification_run_id": "run_42d33d7967059d3e8f22", "domain_scores": {"agriculture,